# Progressive Duplicate Detection

## Package imports & loading the data

In [ ]:
import io

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import scienceplots  # noqa: F401 used for styling the plots (can be commented out if hard to set up)
from IPython.display import clear_output

from ppsnm import ProgressiveEvaluation
from ppsnm.baselines.slorp import SortedListOfRecordPairs
from ppsnm.baselines.snm import SortedNeighbourhoodMethod
from ppsnm.progressive.blocking import ProgressiveBlocking
from ppsnm.progressive.sorted_neighbourhood import ProgressiveSortedNeighbourhoodMethod

plt.style.use(["science", "nature", "grid", "no-latex"])  # note on scienceplots ^^^

cd = pd.read_csv(
    io.StringIO(
        requests.get(
            "https://www13.hpi.uni-potsdam.de/fileadmin/user_upload/fachgebiete/naumann/projekte/dude/cd.csv",
            verify=False,  # download url does not have a valid SSL certificate in this case
        ).text
    ),
    sep=";",
    quotechar='"',
    escapechar="\\",
    encoding="utf-8",
    on_bad_lines="warn",
)

cd_gs = pd.read_csv(
    io.StringIO(requests.get("https://nextcloud.hpi.de/s/pws7rsJXiWq4Foa/download/cd_gold.csv").text),
    sep=";",
    quotechar='"',
    escapechar="\\",
    encoding="utf-8",
    on_bad_lines="warn",
)

# convert csv to pairs of duplicates in the same order (min, max) the pairs get normalised to in the functions
pk_to_pos = {pk: idx for idx, pk in enumerate(cd["pk"].to_numpy())}
# convert to positional to avoid having to compare primary keys within the algorithms' loops
td = {
    (min(p1, p2), max(p1, p2))
    for _, r in cd_gs.iterrows()
    if (p1 := pk_to_pos.get(r.iloc[0])) is not None and (p2 := pk_to_pos.get(r.iloc[1])) is not None
}
print(f"records: {len(cd)}\n gold rows: {len(cd_gs)}\n mapped unique pairs: {len(td)}")

clear_output()

## Defining & running the methods

In [ ]:
keys = ["track01"]
cd[keys] = cd[keys].map(ProgressiveEvaluation._clean_str)  # normalises sorting key

attributes = ["artist", "track01"]
p_size = 100_000  # mirrors the paper's approximate record-buffer size. all data is in memory.
psnm = ProgressiveSortedNeighbourhoodMethod(
    D=cd, K=keys, W=20, I=1, attributes=attributes, true_duplicates=td, pSize=p_size, transitive_closure=True
)
psnm.fit()

pb = ProgressiveBlocking(
    D=cd, K=keys, S=5, R=3, attributes=attributes, true_duplicates=td, pSize=p_size, transitive_closure=True
)
pb.fit()

snm = SortedNeighbourhoodMethod(D=cd, K=keys, W=20, attributes=attributes, true_duplicates=td, transitive_closure=True)
snm.fit()

slorp = SortedListOfRecordPairs(
    D=cd, K=keys, W=20, attributes=attributes, true_duplicates=td, pSize=p_size, transitive_closure=True
)
slorp.fit()

## Progressive quality of the methods

In [ ]:
Q_INTERVALS = 10  # unit of t for fluctuating / timeslot / pipeline
ECONOMIC_INTERVALS = 100  # sampling resolution of the economic weighting

weightings = ("economic", "fluctuating", "timeslot", "pipeline")
q_runs = {"PSNM": psnm, "PB": pb, "SLORP": slorp, "SNM": snm}
n_gold = len(td)
T = ProgressiveEvaluation.common_t(q_runs, x="comparisons")
unit = T / Q_INTERVALS
L_slot = T / 2

q_values = {
    w: ProgressiveEvaluation.q_table(
        q_runs,
        N=n_gold,
        weighting=w,
        x="comparisons",
        T=T,
        n_intervals=ECONOMIC_INTERVALS if w == "economic" else Q_INTERVALS,
        L=L_slot if w == "timeslot" else None,  # fixed slot length in comparisons
    )
    for w in weightings
}

print(f"Q at common T = {T:.0f} comparisons (unit = {unit:.0f})")
print(f"{'weighting':<12}" + "".join(f"{name:>8}" for name in q_runs))
for w in weightings:
    print(f"{w:<12}" + "".join(f"{q_values[w][name]:>8.1%}" for name in q_runs))

## Performance against baselines

In [ ]:
def with_end(alg):  # pads all plots to the same length to show convergenge to the same point
    m = alg.performance
    end = m.iloc[-1].copy()
    end["elapsed_seconds"] = alg.runtime
    end["comparisons"] = alg.total_comparisons
    return pd.concat([m, end.to_frame().T], ignore_index=True)


runs = {"PSNM": with_end(psnm), "PB": with_end(pb), "SLORP": with_end(slorp), "SNM": with_end(snm)}
colors = {"PSNM": "C5", "PB": "C3", "SLORP": "C4", "SNM": "C0"}
markers = {"PSNM": "x", "PB": "^", "SLORP": "+", "SNM": "s"}

fig, axes = plt.subplots(3, 1, figsize=(7, 6), dpi=300, constrained_layout=True, sharex=True)
fig.suptitle("CD dataset")

for name, metrics in runs.items():
    comparisons = metrics["comparisons"].to_numpy()
    discovered = metrics["duplicates_found"].to_numpy(dtype=float)
    style = {
        "color": colors[name],
        "marker": markers[name],
        "markerfacecolor": "white",
        "markeredgecolor": colors[name],
        "markevery": 35,
    }
    yield_per_comparison = np.divide(
        discovered, comparisons, out=np.full_like(discovered, np.nan), where=comparisons > 0
    )
    axes[0].plot(comparisons, yield_per_comparison, label=name, **style)
    prec = metrics["precision"].where(metrics["duplicates_found"] >= 10)
    axes[1].plot(comparisons, prec, color=colors[name])
    axes[1].plot(comparisons, metrics["recall"], color=colors[name], linestyle="--")
    axes[2].plot(comparisons, discovered, label=name, **style)

axes[0].set(title="Pair yield", ylabel="Duplicates found / comparisons")
axes[1].set(title="Precision and recall", ylabel="Score", ylim=(0, 1), yticks=np.linspace(0, 1, 6))
axes[2].set(title="Duplicates found over comparisons", xlabel="Number of comparisons", ylabel="Duplicates found")
for ax in axes:
    ax.grid(axis="x")

axes[1].legend(  # plot extra legend
    handles=[
        plt.Line2D([], [], color="black", linestyle=style, label=label)
        for style, label in [("-", "Precision"), ("--", "Recall")]
    ],
    loc="lower right",
    frameon=False,
)
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside lower center", ncol=len(runs), frameon=False)